In [2]:
import scipy.io
import xarray as xr
import numpy as np

In [3]:
# Load .mat file
mat_data = scipy.io.loadmat('Isafjardardjup_hypsography/Isafjardardjup_hypso_tmp_v20190117.mat')

In [4]:
# Remove MATLAB metadata fields
mat_data_clean = {k: v for k, v in mat_data.items() if not k.startswith('__')}

# Display variable shapes to handle dimensions properly
var_shapes = {k: v.shape for k, v in mat_data_clean.items()}
var_shapes

{'e': (1, 1)}

In [5]:
e_contents = mat_data_clean['e']

In [6]:
# Extract the struct from the array
e_struct = e_contents[0, 0]

# Extract all fields and their shapes/types
field_info = {name: (type(e_struct[name]), np.shape(e_struct[name])) for name in e_struct.dtype.names}
field_info

{'dlon': (numpy.ndarray, (1, 1)),
 'dlat': (numpy.ndarray, (1, 1)),
 'lonv': (numpy.ndarray, (1, 641)),
 'latv': (numpy.ndarray, (1, 651)),
 'lon': (numpy.ndarray, (651, 641)),
 'lat': (numpy.ndarray, (651, 641)),
 'dep': (numpy.ndarray, (651, 641)),
 'depfj': (numpy.ndarray, (651, 641)),
 'areakm2': (numpy.ndarray, (651, 641)),
 'stat': (numpy.ndarray, (1, 1)),
 'sill1': (numpy.ndarray, (1, 1))}

In [7]:
# Helper to extract scalar or array from nested structure
def extract_value(obj):
    if isinstance(obj, np.ndarray) and obj.size == 1:
        return obj.item()
    return obj.squeeze()

In [8]:
# Extract data
dlon = extract_value(e_struct['dlon'])
dlat = extract_value(e_struct['dlat'])
lonv = extract_value(e_struct['lonv'])  # shape (641,)
latv = extract_value(e_struct['latv'])  # shape (651,)
lon = extract_value(e_struct['lon'])    # shape (651, 641)
lat = extract_value(e_struct['lat'])    # shape (651, 641)
dep = extract_value(e_struct['dep'])    # shape (651, 641)
depfj = extract_value(e_struct['depfj'])  # shape (651, 641)
areakm2 = extract_value(e_struct['areakm2'])  # shape (651, 641)
stat = extract_value(e_struct['stat'])
sill1 = extract_value(e_struct['sill1'])

In [10]:
lon.min()

np.float64(-23.58)

In [22]:
# Create the Dataset with valid coordinates and variables
ds = xr.Dataset(
    {
        "depth": (("y", "x"), dep),
        "depth_fjord": (("y", "x"), depfj),
        "area_km2": (("y", "x"), areakm2),
        "lon_grid": (("y", "x"), lon),
        "lat_grid": (("y", "x"), lat),
    },
    coords={
        "lonv": ("x", lonv),
        "latv": ("y", latv),
        "dlon": dlon,
        "dlat": dlat,
    },
    attrs={
        "stat_summary": str(stat),
        "sill1_info": str(sill1),
    }
)

In [23]:
# Save to NetCDF
output_path = "Isafjardardjup_topo.nc"
ds.to_netcdf(output_path)